# Chapter 8 - Linear Models

In this chapter we start working with linear models through Linear Regression.
The goal is to understand what a linear model assumes, how Ordinary Least
Squares (OLS) finds the best-fitting line directly, and how Gradient Descent
reaches a similar solution through repeated updates.

We will finish with the Advertising dataset and compare two scikit-learn
implementations:

- LinearRegression, which uses the OLS idea.
- SGDRegressor, which learns with gradient-descent-style updates.


## 1. Linear And Non-Linear Models

A linear model assumes that the target changes in a proportional and additive
way as the input features change. In two dimensions this looks like a line. In
higher dimensions it becomes a plane or hyperplane.

A non-linear model does not need to follow a straight-line relationship. It can
bend, split, or curve to capture more complex patterns.

The next small example gives us a concrete feel for this difference.


In [1]:
# import all the required libraries
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, SGDRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# A simple teaching dataset: hotter days usually sell more ice cream.
ice_cream_sales = pd.DataFrame(
    {
        "temperature": [20, 22, 24, 26, 28, 30, 32, 34, 36, 38],
        "sales": [145, 160, 172, 185, 201, 215, 230, 244, 259, 275],
    }
)

print("Simple linear-looking data:")
print(ice_cream_sales)


Simple linear-looking data:
   temperature  sales
0           20    145
1           22    160
2           24    172
3           26    185
4           28    201
5           30    215
6           32    230
7           34    244
8           36    259
9           38    275


The values rise in a steady way. This is the kind of pattern where a linear
model is a natural first choice.


In [2]:
# Fit a one-feature linear regression model to the teaching dataset.
X_simple = ice_cream_sales[["temperature"]]
y_simple = ice_cream_sales["sales"]

simple_model = LinearRegression()
simple_model.fit(X_simple, y_simple)

print(f"Slope (m): {simple_model.coef_[0]:.2f}")
print(f"Intercept (b): {simple_model.intercept_:.2f}")

# Predict sales for one new temperature value.
new_temperature = pd.DataFrame({"temperature": [35]})
predicted_sales = simple_model.predict(new_temperature)[0]
print(f"Predicted sales at 35 degrees: {predicted_sales:.1f}")


Slope (m): 7.19
Intercept (b): 0.15
Predicted sales at 35 degrees: 251.7


The fitted model has the form y = mx + b. 

The slope tells us how much the prediction changes when temperature increases by one unit.

The intercept is the model's baseline prediction when the feature value is zero.


## 2. Geometric Intuition For Linear Regression

Linear Regression searches for the line that best represents the relationship
between the inputs and the target. 
- With one input feature, this is a line. 
- With multiple input features, the same idea extends to a plane or hyperplane.

The model makes predictions by combining feature values with learned weights:

y = w1*x1 + w2*x2 + ... + wn*xn + b

The weights show how strongly each feature contributes to the prediction.


## 3. Ordinary Least Squares (OLS)

OLS chooses the line that minimizes the squared errors between actual and
predicted values. Squaring is useful because negative and positive errors do not
cancel each other out, and larger mistakes receive a larger penalty.

First, let us intentionally start with a poor line so the error calculation is
visible.


In [3]:
# Before: use a random line from the chapter discussion.
ols_demo = ice_cream_sales.copy()
ols_demo["random_prediction"] = 10 * ols_demo["temperature"] + 50
ols_demo["random_error"] = ols_demo["sales"] - ols_demo["random_prediction"]
ols_demo["random_squared_error"] = ols_demo["random_error"] ** 2

print("Before OLS: errors from a random line")
print(ols_demo.head())


Before OLS: errors from a random line
   temperature  sales  random_prediction  random_error  random_squared_error
0           20    145                250          -105                 11025
1           22    160                270          -110                 12100
2           24    172                290          -118                 13924
3           26    185                310          -125                 15625
4           28    201                330          -129                 16641


In [4]:
# After: let LinearRegression find the best OLS line.
ols_demo["ols_prediction"] = simple_model.predict(X_simple)
ols_demo["ols_error"] = ols_demo["sales"] - ols_demo["ols_prediction"]
ols_demo["ols_squared_error"] = ols_demo["ols_error"] ** 2

random_sse = ols_demo["random_squared_error"].sum()
ols_sse = ols_demo["ols_squared_error"].sum()

print(f"Random line SSE: {random_sse:.2f}")
print(f"OLS line SSE: {ols_sse:.2f}")
print(ols_demo[["temperature", "sales", "random_prediction", "ols_prediction"]].head())


Random line SSE: 175282.00
OLS line SSE: 12.75
   temperature  sales  random_prediction  ols_prediction
0           20    145                250      143.909091
1           22    160                270      158.284848
2           24    172                290      172.660606
3           26    185                310      187.036364
4           28    201                330      201.412121


The OLS line produces a much smaller sum of squared errors. 

This is the central idea of OLS: find the values of slope and intercept that make total squared
error as small as possible.


## 4. Gradient Descent

OLS gives a direct mathematical solution, but direct solutions can become
expensive when the data has many features. Gradient Descent solves the same
kind of optimization problem iteratively.

The workflow is:

1. Start with initial parameter values.
2. Use a cost function such as Mean Squared Error to measure how bad the
   current predictions are.
3. Compute the gradient, which tells us how to change the parameters.
4. Multiply the gradient by a learning rate so the update is controlled.
5. Stop after convergence or after a fixed number of iterations.

The next cell implements simple batch gradient descent for the one-feature ice
cream example. This is for intuition before we use scikit-learn.


In [5]:
# Before: start from a flat line with zero slope and zero intercept.
x = ice_cream_sales["temperature"].to_numpy(dtype=float)
y = ice_cream_sales["sales"].to_numpy(dtype=float)

# Standardizing x keeps the updates stable for this demonstration.
x_scaled = (x - x.mean()) / x.std()

m = 0.0
b = 0.0
learning_rate = 0.01
n = len(x_scaled)
history = []

for iteration in range(1, 1001):
    y_pred = m * x_scaled + b
    error = y_pred - y
    mse = np.mean(error ** 2)

    gradient_m = (2 / n) * np.sum(error * x_scaled)
    gradient_b = (2 / n) * np.sum(error)

    m = m - learning_rate * gradient_m
    b = b - learning_rate * gradient_b

    if iteration in [1, 2, 5, 10, 100, 500, 1000]:
        history.append({"iteration": iteration, "m": m, "b": b, "mse": mse})

gd_history = pd.DataFrame(history)
print("Gradient descent progress:")
print(gd_history)


Gradient descent progress:
   iteration          m           b           mse
0          1   0.825824    4.172000  45220.200000
1          2   1.635132    8.260560  43429.530576
2          5   3.967228   20.042122  38471.864334
3         10   7.553287   38.158612  31434.547437
4        100  35.815197  180.935561    829.375980
5        500  41.289526  208.591442      1.275231
6       1000  41.291220  208.600000      1.275152


In [6]:
# After: use the final parameters to check the fitted predictions.
gd_demo = ice_cream_sales.copy()
gd_demo["gd_prediction"] = m * x_scaled + b

print(f"Final gradient descent MSE: {mean_squared_error(y, gd_demo['gd_prediction']):.2f}")
print(gd_demo.head())


Final gradient descent MSE: 1.28
   temperature  sales  gd_prediction
0           20    145     143.909091
1           22    160     158.284848
2           24    172     172.660606
3           26    185     187.036363
4           28    201     201.412121


The cost falls as the iterations continue. Gradient Descent is not directly
solving the OLS formula; it is moving step by step toward parameters that give
low prediction error.


## 5. Gradient Descent Variants

The chapter describes three common ways to choose data for each update:

- Batch Gradient Descent uses the full dataset for each update. It is stable,
  but expensive for large datasets.
- Stochastic Gradient Descent uses one random row for each update. It is fast,
  but its path can be noisy.
- Mini-Batch Gradient Descent uses a small group of rows for each update. It is
  a practical balance between the first two approaches.

In scikit-learn, SGDRegressor gives us an implementation based on stochastic
gradient descent.


## 6. Code Example - Advertising Sales Prediction

Now we apply the chapter's practical example. The Advertising dataset contains
spending on TV, radio, and newspaper advertising, along with product sales.

We will train two models:

- OLS through LinearRegression.
- Gradient Descent through SGDRegressor.


In [7]:
# Load the required data

url = 'https://raw.githubusercontent.com/kanetkar/LULML/refs/heads/main/ch08/Advertising.csv'

advertising = pd.read_csv(url)

print("First five rows of the Advertising dataset:")
print(advertising.head())


First five rows of the Advertising dataset:
   ID     TV  Radio  Newspaper  Sales
0   1  230.1   37.8       69.2   22.1
1   2   44.5   39.3       45.1   10.4
2   3   17.2   45.9       69.3    9.3
3   4  151.5   41.3       58.5   18.5
4   5  180.8   10.8       58.4   12.9


In [8]:
print("Dataset shape:", advertising.shape)
print("Missing values by column:")
print(advertising.isnull().sum())


Dataset shape: (200, 5)
Missing values by column:
ID           0
TV           0
Radio        0
Newspaper    0
Sales        0
dtype: int64


There are no missing values in this dataset. The first column is an ID column,
so it should not be used as a predictive feature.


In [9]:
# Before: inspect all columns before choosing features and target.
print("Columns in the raw dataset:")
print(advertising.columns.tolist())

feature_columns = ["TV", "Radio", "Newspaper"]
target_column = "Sales"

X = advertising[feature_columns]
y = advertising[target_column]

print("\nFeature preview:")
print(X.head())


Columns in the raw dataset:
['ID', 'TV', 'Radio', 'Newspaper', 'Sales']

Feature preview:
      TV  Radio  Newspaper
0  230.1   37.8       69.2
1   44.5   39.3       45.1
2   17.2   45.9       69.3
3  151.5   41.3       58.5
4  180.8   10.8       58.4


In [10]:
print("Target preview:")
print(y.head())


Target preview:
0    22.1
1    10.4
2     9.3
3    18.5
4    12.9
Name: Sales, dtype: float64


## 7. Train-Test Split

We split the data so the model learns on one part and is evaluated on unseen
rows. The chapter uses 80 percent for training and 20 percent for testing.


In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

print("Training feature shape:", X_train.shape)
print("Test feature shape:", X_test.shape)


Training feature shape: (160, 3)
Test feature shape: (40, 3)


## 8. Standardizing The Features

Gradient-based models are easier to train when features have a similar scale.
We fit the scaler only on the training data, then apply the same learned
transformation to the test data.


In [12]:
# Before: the original feature scales are different.
print("Before scaling:")
print(X_train.head())


Before scaling:
        TV  Radio  Newspaper
79   116.0    7.7       23.1
197  177.0    9.3        6.4
38    43.1   26.7       35.1
24    62.3   12.6       18.3
122  224.0    2.4       15.6


In [13]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=feature_columns, index=X_train.index)

print("After scaling:")
print(X_train_scaled_df.head())


After scaling:
           TV     Radio  Newspaper
79  -0.404248 -1.028237  -0.337675
197  0.320608 -0.919828  -1.161439
38  -1.270511  0.259124   0.254251
24  -1.042359 -0.696233  -0.574446
122  0.879103 -1.387343  -0.707629


In [14]:
print("Scaled training mean, rounded:")
print(X_train_scaled_df.mean().round(4))

print("\nScaled training standard deviation, rounded:")
print(X_train_scaled_df.std(ddof=0).round(4))


Scaled training mean, rounded:
TV          -0.0
Radio       -0.0
Newspaper    0.0
dtype: float64

Scaled training standard deviation, rounded:
TV           1.0
Radio        1.0
Newspaper    1.0
dtype: float64


The scaled training features now have mean close to 0 and standard deviation
close to 1. We use these scaled arrays for both models so the comparison is
consistent.


## 9. Linear Regression With OLS

LinearRegression learns the best coefficients for the squared-error objective
using the OLS idea.


In [15]:
ols_model = LinearRegression()
ols_model.fit(X_train_scaled, y_train)

y_pred_ols = ols_model.predict(X_test_scaled)

mse_ols = mean_squared_error(y_test, y_pred_ols)
r2_ols = r2_score(y_test, y_pred_ols)

print(f"OLS MSE: {mse_ols:.2f}")
print(f"OLS R2: {r2_ols:.2f}")


OLS MSE: 3.17
OLS R2: 0.90


In [16]:
ols_coefficients = pd.DataFrame(
    {
        "feature": feature_columns,
        "coefficient": ols_model.coef_,
    }
)

print("OLS intercept:", round(ols_model.intercept_, 4))
print(ols_coefficients)


OLS intercept: 14.1
     feature  coefficient
0         TV     3.764196
1      Radio     2.792307
2  Newspaper     0.055976


The coefficients are learned on scaled features, so their sizes are comparable.

TV and Radio have stronger positive contributions than Newspaper for this dataset.


## 10. Linear Regression With Gradient Descent

SGDRegressor adjusts the coefficients iteratively. We use squared error loss - so
it is optimizing the same kind of objective as Linear Regression.


In [17]:
gd_model = SGDRegressor(
    loss="squared_error",
    learning_rate="constant",
    eta0=0.01,
    max_iter=1000,
    random_state=42,
)

gd_model.fit(X_train_scaled, y_train)
y_pred_gd = gd_model.predict(X_test_scaled)

mse_gd = mean_squared_error(y_test, y_pred_gd)
r2_gd = r2_score(y_test, y_pred_gd)

print(f"GD MSE: {mse_gd:.2f}")
print(f"GD R2: {r2_gd:.2f}")


GD MSE: 3.19
GD R2: 0.90


In [18]:
gd_coefficients = pd.DataFrame(
    {
        "feature": feature_columns,
        "coefficient": gd_model.coef_,
    }
)

print("Gradient Descent intercept:", round(gd_model.intercept_[0], 4))
print(gd_coefficients)


Gradient Descent intercept: 14.0611
     feature  coefficient
0         TV     3.868805
1      Radio     2.775730
2  Newspaper     0.150258


## 11. Comparing OLS And Gradient Descent

Both methods are trying to minimize prediction error. OLS reaches the solution
directly, while Gradient Descent approaches a good solution through repeated
parameter updates.


In [19]:
comparison = pd.DataFrame(
    {
        "model": ["OLS LinearRegression", "Gradient Descent SGDRegressor"],
        "mse": [mse_ols, mse_gd],
        "r2": [r2_ols, r2_gd],
    }
)

print(comparison)


                           model       mse        r2
0           OLS LinearRegression  3.174097  0.899438
1  Gradient Descent SGDRegressor  3.187520  0.899013


In [20]:
prediction_check = pd.DataFrame(
    {
        "actual_sales": y_test.to_numpy(),
        "ols_prediction": y_pred_ols,
        "gd_prediction": y_pred_gd,
    },
    index=y_test.index,
)

prediction_check["ols_error"] = prediction_check["actual_sales"] - prediction_check["ols_prediction"]
prediction_check["gd_error"] = prediction_check["actual_sales"] - prediction_check["gd_prediction"]

print(prediction_check.head())


     actual_sales  ols_prediction  gd_prediction  ols_error  gd_error
95           16.9       16.408024      16.482586   0.491976  0.417414
15           22.4       20.889882      20.986263   1.510118  1.413737
30           21.4       21.553843      21.748100  -0.153843 -0.348100
158           7.3       10.608503      10.452854  -3.308503 -3.152854
128          24.7       22.112373      22.007107   2.587627  2.692893


The two models produce very similar scores. Small differences are expected
because the gradient descent model reaches the solution iteratively, while OLS
computes its solution directly.


## 12. Chapter Recap

In this chapter we learned that:

- Linear models assume an additive relationship between features and target.
- Linear Regression fits a line, plane, or hyperplane depending on the number
  of input features.
- OLS finds coefficients by minimizing the sum of squared errors directly.
- Gradient Descent minimizes a cost function through repeated updates.
- Batch, stochastic, and mini-batch gradient descent differ in how much data
  they use for each update.
- On the Advertising dataset, OLS and SGD-based Linear Regression give nearly
  the same predictive performance when the SGD model converges well.
